In [8]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# Display / plotting defaults
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.dpi"] = 110

RAW_DATA_PATH = Path("default of credit card clients - default of credit card clients.csv")
CLEAN_DATA_PATH = Path("cleaned_credit_data.csv")
DATA_DICTIONARY_PATH = Path("data_dictionary_cleaned.csv")

RANDOM_STATE = 42
print("Environment ready.")


Environment ready.


In [9]:
def load_raw_data(path: Path) -> pd.DataFrame:
    """Load the raw credit-card dataset exactly as supplied and validate its shape.

    The supplied CSV has a known non-standard layout: an extra generic
    ``X1, X2, ..., Y`` row precedes the real header row. ``header=1`` skips
    that artefact row and uses the true column names on row 2.

    Parameters
    ----------
    path : Path
        Location of the raw CSV file.

    Returns
    -------
    pd.DataFrame
        The raw, unmodified dataset (real column names, no artefact row).

    Raises
    ------
    FileNotFoundError
        If the file does not exist at the given path.
    AssertionError
        If the loaded shape does not match the documented 30,000 x 25 dataset.
    """
    if not path.exists():
        raise FileNotFoundError(f"Raw data file not found at: {path.resolve()}")

    df_raw = pd.read_csv(path, header=1)

    expected_rows, expected_cols = 30_000, 25
    assert df_raw.shape == (expected_rows, expected_cols), (
        f"Unexpected shape {df_raw.shape}; expected ({expected_rows}, {expected_cols}). "
        "Verify the source file matches the UCI 'Default of Credit Card Clients' dataset."
    )
    return df_raw


df_raw = load_raw_data(RAW_DATA_PATH)
print(f"Source file: {RAW_DATA_PATH.name}")
print(f"Loaded {df_raw.shape[0]:,} rows and {df_raw.shape[1]} columns.")
print(f"Row/column count matches UCI documentation: {df_raw.shape == (30000, 25)}")
df_raw.head()


Source file: default of credit card clients - default of credit card clients.csv
Loaded 30,000 rows and 25 columns.
Row/column count matches UCI documentation: True


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [10]:
df_raw.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   ID                          30000 non-null  int64
 1   LIMIT_BAL                   30000 non-null  int64
 2   SEX                         30000 non-null  int64
 3   EDUCATION                   30000 non-null  int64
 4   MARRIAGE                    30000 non-null  int64
 5   AGE                         30000 non-null  int64
 6   PAY_1                       30000 non-null  int64
 7   PAY_2                       30000 non-null  int64
 8   PAY_3                       30000 non-null  int64
 9   PAY_4                       30000 non-null  int64
 10  PAY_5                       30000 non-null  int64
 11  PAY_6                       30000 non-null  int64
 12  BILL_AMT1                   30000 non-null  int64
 13  BILL_AMT2                   30000 non-null  int64
 14  BILL_A

In [11]:
df_raw.describe(include="all").T


,count,mean,std,min,25%,50%,75%,max
ID,30000.0,15000.500000,8660.398374,1.0,7500.75,15000.5,22500.25,30000.0
LIMIT_BAL,30000.0,167484.322667,129747.661567,10000.0,50000.00,140000.0,240000.00,1000000.0
SEX,30000.0,1.603733,0.489129,1.0,1.00,2.0,2.00,2.0
EDUCATION,30000.0,1.853133,0.790349,0.0,1.00,2.0,2.00,6.0
MARRIAGE,30000.0,1.551867,0.521970,0.0,1.00,2.0,2.00,3.0
AGE,30000.0,35.485500,9.217904,21.0,28.00,34.0,41.00,79.0
PAY_1,30000.0,-0.016700,1.123802,-2.0,-1.00,0.0,0.00,8.0
PAY_2,30000.0,-0.133767,1.197186,-2.0,-1.00,0.0,0.00,8.0
PAY_3,30000.0,-0.166200,1.196868,-2.0,-1.00,0.0,0.00,8.0
PAY_4,30000.0,-0.220667,1.169139,-2.0,-1.00,0.0,0.00,8.0


In [12]:
missing_summary = (
    df_raw.isnull()
    .sum()
    .to_frame("missing_count")
    .assign(missing_pct=lambda d: (d["missing_count"] / len(df_raw) * 100).round(3))
    .sort_values("missing_count", ascending=False)
)
print("Columns with missing values:", (missing_summary["missing_count"] > 0).sum())
missing_summary.head(10)


Columns with missing values: 0


,missing_count,missing_pct
ID,0,0.0
LIMIT_BAL,0,0.0
SEX,0,0.0
EDUCATION,0,0.0
MARRIAGE,0,0.0
AGE,0,0.0
PAY_1,0,0.0
PAY_2,0,0.0
PAY_3,0,0.0
PAY_4,0,0.0


In [13]:
print("Raw SEX codes:\n", df_raw["SEX"].value_counts(dropna=False), "\n")
print("Raw EDUCATION codes:\n", df_raw["EDUCATION"].value_counts(dropna=False), "\n")
print("Raw MARRIAGE codes:\n", df_raw["MARRIAGE"].value_counts(dropna=False), "\n")

pay_cols_raw = [c for c in df_raw.columns if c.startswith("PAY_") and not c.startswith("PAY_AMT")]
print("Repayment-status columns:", pay_cols_raw)
for c in pay_cols_raw:
    print(f"\n{c} value counts:\n", df_raw[c].value_counts().sort_index())


Raw SEX codes:
 SEX
2    18112
1    11888
Name: count, dtype: int64 

Raw EDUCATION codes:
 EDUCATION
2    14030
1    10585
3     4917
5      280
4      123
6       51
0       14
Name: count, dtype: int64 

Raw MARRIAGE codes:
 MARRIAGE
2    15964
1    13659
3      323
0       54
Name: count, dtype: int64 

Repayment-status columns: ['PAY_1', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']

PAY_1 value counts:
 PAY_1
-2     2759
-1     5686
 0    14737
 1     3688
 2     2667
 3      322
 4       76
 5       26
 6       11
 7        9
 8       19
Name: count, dtype: int64

PAY_2 value counts:
 PAY_2
-2     3782
-1     6050
 0    15730
 1       28
 2     3927
 3      326
 4       99
 5       25
 6       12
 7       20
 8        1
Name: count, dtype: int64

PAY_3 value counts:
 PAY_3
-2     4085
-1     5938
 0    15764
 1        4
 2     3819
 3      240
 4       76
 5       21
 6       23
 7       27
 8        3
Name: count, dtype: int64

PAY_4 value counts:
 PAY_4
-2     4348
-1     5687


In [14]:
def build_rename_map(columns: list[str]) -> dict[str, str]:
    """Construct a mapping from the supplied raw column names to clean snake_case names.

    Note: in the classic UCI export the most recent repayment-status column is
    named ``PAY_0``; in this project's supplied file it is already named
    ``PAY_1``, so no renumbering quirk needs to be handled here — we map it
    directly.
    """
    rename_map = {
        "ID": "customer_id",
        "LIMIT_BAL": "limit_balance",
        "SEX": "sex",
        "EDUCATION": "education",
        "MARRIAGE": "marriage",
        "AGE": "age",
        "default payment next month": "default_next_month",
    }

    # PAY_1..PAY_6 -> pay_status_1..pay_status_6
    for month in range(1, 7):
        rename_map[f"PAY_{month}"] = f"pay_status_{month}"

    for month in range(1, 7):
        rename_map[f"BILL_AMT{month}"] = f"bill_amount_{month}"
        rename_map[f"PAY_AMT{month}"] = f"pay_amount_{month}"

    missing = set(rename_map) - set(columns)
    if missing:
        raise KeyError(f"Expected raw columns not found in dataframe: {missing}")

    return rename_map


rename_map = build_rename_map(list(df_raw.columns))
df = df_raw.rename(columns=rename_map)

print("Renamed columns:")
print(list(df.columns))


Renamed columns:
['customer_id', 'limit_balance', 'sex', 'education', 'marriage', 'age', 'pay_status_1', 'pay_status_2', 'pay_status_3', 'pay_status_4', 'pay_status_5', 'pay_status_6', 'bill_amount_1', 'bill_amount_2', 'bill_amount_3', 'bill_amount_4', 'bill_amount_5', 'bill_amount_6', 'pay_amount_1', 'pay_amount_2', 'pay_amount_3', 'pay_amount_4', 'pay_amount_5', 'pay_amount_6', 'default_next_month']


In [15]:
SEX_LABELS = {1: "Male", 2: "Female"}

# Codes 0, 5, 6 are not defined in the UCI documentation (only 1-4 are documented).
EDUCATION_LABELS = {
    1: "Graduate School",
    2: "University",
    3: "High School",
    4: "Others",
    0: "Unknown/Other",
    5: "Unknown/Other",
    6: "Unknown/Other",
}

# Code 0 is not defined in the UCI documentation (only 1-3 are documented).
MARRIAGE_LABELS = {
    1: "Married",
    2: "Single",
    3: "Others",
    0: "Unknown/Other",
}


def decode_categoricals(frame: pd.DataFrame) -> pd.DataFrame:
    """Return a copy of ``frame`` with sex/education/marriage decoded to labels.

    Undocumented numeric codes are mapped to an explicit 'Unknown/Other' label
    (never dropped, never silently merged into a real category) so downstream
    consumers can see exactly how much of the portfolio has ambiguous
    demographic data.
    """
    out = frame.copy()

    out["sex_label"] = out["sex"].map(SEX_LABELS)
    out["education_label"] = out["education"].map(EDUCATION_LABELS)
    out["marriage_label"] = out["marriage"].map(MARRIAGE_LABELS)

    for col in ["sex_label", "education_label", "marriage_label"]:
        n_unmapped = out[col].isnull().sum()
        if n_unmapped:
            raise ValueError(f"{n_unmapped} unmapped codes remain in {col}; extend the mapping.")

    return out


df = decode_categoricals(df)

print("Education label distribution (post-decoding):")
print(df["education_label"].value_counts(), "\n")
print("Marriage label distribution (post-decoding):")
print(df["marriage_label"].value_counts(), "\n")

n_edu_unknown = (df["education_label"] == "Unknown/Other").sum()
n_mar_unknown = (df["marriage_label"] == "Unknown/Other").sum()
print(f"Accounts with undocumented education code: {n_edu_unknown} "
      f"({n_edu_unknown / len(df):.2%} of portfolio)")
print(f"Accounts with undocumented marriage code:  {n_mar_unknown} "
      f"({n_mar_unknown / len(df):.2%} of portfolio)")


Education label distribution (post-decoding):
education_label
University         14030
Graduate School    10585
High School         4917
Unknown/Other        345
Others               123
Name: count, dtype: int64 

Marriage label distribution (post-decoding):
marriage_label
Single           15964
Married          13659
Others             323
Unknown/Other       54
Name: count, dtype: int64 

Accounts with undocumented education code: 345 (1.15% of portfolio)
Accounts with undocumented marriage code:  54 (0.18% of portfolio)
